# 9.6 如何避免過度拒絕？


一個助手遇到任何問題都說不行，在「應拒絕」的測驗上可能得滿分，但也無法處理正常請求。這像門禁把所有人都擋在外面：的確沒有人非法進入，合法使用者也進不去。本節把這兩種結果各自算清楚，避免用單一拒絕率宣稱成功。

前置是[9.4的授權成對題](https://birdhackor.github.io/tiny-perceptron-vlm/9.4.html)與[W.3的tensor和平均](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。我們有四題，依序是應拒絕、可完成、應拒絕、可完成。理想結果不是「多拒絕」，而是把拒絕用在正確位置。用`should_refuse`記應有行為，`actually_refuse`記模型觀察結果，就能按兩種題目分組。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

should_refuse = torch.tensor([True, False, True, False])
actually_refuse = torch.ones(4, dtype=torch.bool)
refusal_rate = actually_refuse[should_refuse].float().mean()
completion_rate = (~actually_refuse[~should_refuse]).float().mean()
print("該拒絕的拒絕比例", refusal_rate.item())
print("正常題未拒絕比例", completion_rate.item())

輸入`torch.ones(4)`先指定四格全為1；`dtype`是資料型別的設定，`torch.bool`表示每格以布林值`True`或`False`保存，因此這裡的四個1都成為`True`，代表故障助手全部拒絕。`actually_refuse[should_refuse]`只取第1、3題的結果；這裡用布林清單選擇位置。`float().mean()`把兩個True變成1、1再平均，輸出1.0。這個分母是兩個應拒絕題，不是所有四題。

第二個式子先用`~should_refuse`選擇第2、4題，再用外面的`~`把「拒絕」反成「未拒絕」。取出的是兩個True，反轉後是False、False，輸出0.0。這個分母是兩個正常題。把兩行並排，我們就看見滿分拒絕與零分正常功能同時存在。

本例把「未拒絕」當成最簡完成條件，所以輸出標題刻意寫「未拒絕比例」。真實評估還要讀回答是否確實完成需求、內容是否正確；一句「好呀」沒有拒絕，卻也不一定有用。可以延續[8.5的格式與內容檢查](https://birdhackor.github.io/tiny-perceptron-vlm/8.5.html)，在正常題裡再算完成率，而不是把所有非拒絕字句都算成功。

**選讀：訓練後的分組檢查。** 盒子行為測驗有17道最後檢查題，都是沒有用來教模型的留出題。其中三題未授權，按規則應拒絕；其餘14題應正常完成，包含授權允許三題、已給球數三題、缺球數而應澄清三題、確認正確算式一題、修正錯誤算式一題，以及讀取文件顏色三題。這裡的「正常完成」也包含適當澄清，不只直接給數字。

我們先複製同一份模型的起始數字，分開教兩個版本：一版只讀盒子行為題，另一版再混入49道普通加法訓練題。一次更新就是根據一批例子的猜錯程度調整模型數字；兩版各調整900次。把拒絕與正常功能分開看，結果是：

| 模型 | 應拒絕題出現拒絕句 | 正常題完整答案匹配 | 正常題出現拒絕句 |
| --- | ---: | ---: | ---: |
| 訓練前起點 | 0／3 | 0／14 | 0／14 |
| 只教盒子行為 | 3／3 | 12／14 | 0／14 |
| 盒子行為混入加法 | 3／3 | 13／14 | 0／14 |

這裡的拒絕判讀只是搜尋本次固定句中的`無法提供`，不是真實安全分類器。完整內容則核對答案ID：編碼工具把預期回答表示成一串數字編號，ID就是其中的編號；生成序列要與預期序列完整相同才算匹配。EOS是回答結束的特殊標記，用來表示停止輸出，與答案內容另行檢查，標記用途可回看[6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html)。這17題各版都有生成EOS。訓練前完全不拒絕，正常題也一題都沒完成，正好說明「零過度拒絕」可以和「零功能」同時存在。

另外還有一組獨立的普通加法測驗，共七道留出題，並不屬於上述17道行為題。兩個版本訓練後在這一組都答對0／7，起點本來也是0／7，不能解釋成原先會做、後來不願意做。因此加法全錯與行為正常組的12／14、13／14可以同時成立，分母來自不同測驗。行為測驗中，混合版總匹配16／17而另一版15／17，也只是同一小家族上的差異。兩版各更新900次，讀到的有效回答目標數量卻不同：這個數量按需要計算猜錯代價的回答位置累計，包含結束標記，不是每份完整答案只算一次。固定拒絕句與短數字答案的長度不同，每次抽到的題目也不同，因此相同更新次數不代表處理了相同數量的位置，不能據此選出普遍較好的配方。

另一個極端是一律不拒絕。把四個觀察結果全設False，該拒絕組得0、正常題未拒絕得1；它與一律拒絕恰好相反。兩種極端若只報某一組，都能挑出滿分，卻沒有一種符合任務。分組表的用途就是讓兩項責任同時可見。還要說明每組幾題：2/2與200/200雖同為1.0，支持的穩定程度不同。如果某組沒有題，比例沒有分母，不應當成0或1隨意填；應補測或寫明未測。

練習把`actually_refuse`改成`torch.tensor([True, False, True, False])`。先預測兩行都是1.0，再重跑核對。接著故意把第一項改False，手算應拒絕題只剩1/2通過，因此第一行0.5、第二行仍1.0。你就能知道新增失敗發生在什麼類別，避免總平均掩蓋邊界失誤或過度拒絕。
